# Champs2D — post-treatment fields, 2D vector model

`Postraitement_fields` · ASN 28 validation form (TRUST-LiMe).

Merged form: the eleven 2D post-treatment fields are validated on **two
datasets** instead of eleven separate forms. The fields split into two
families that cannot share one physical state:

| Dataset | State | Fields |
|---|---|---|
| `champs2d_gradient` | linear $b=(x-y,\,x+y)$, $\eta=0$, no velocity | `divergence_magnetique` (interior **and** wall cell), `rotationnel_magnetique`, `courant_magnetique`, `force_lorentz`, `tension_magnetique` |
| `champs2d_transport` | uniform $b=(3,4)$, uniform $u=(4,3)$, $\eta=0.25$ | `vecteur_poynting`, `helicite_croisee`, `nombre_de_reynolds_magnetique`, `nombre_de_prandtl_magnetique`, `energie_magnetique`, `pulsation_cyclotronique` |

**Why two datasets and not one.** The five $\nabla b$ fields need $\eta=0$ (a
linear $b$ is then an exact equilibrium, so the analytic reference survives
even at the wall cells), while `nombre_de_reynolds_magnetique` divides by
$\eta$ and therefore needs $\eta>0$. The transport family also needs a
velocity, and a non-zero velocity would advect a non-uniform $b$. Both datasets
keep their own 3-level mesh-convergence ladder.

## Introduction

Validation of the **post-treatment fields** of the standalone magnetic
(induction) equation solved by TRUST-LiMe:

$$\partial_t b + (u \cdot \nabla) b - \nabla \cdot (\eta \nabla b) = 0$$

The fields are created lazily by
`Convection_Diffusion_Magnetic_field::creer_champ()` and evaluated by
`Champ_Fonc_MHD` (`src/Champ_Fonc_MHD.{h,cpp}`) on top of the pure arithmetic
kernels of `src/MHD_posttraitement_kernel.h`. Each `Sondes` entry resolves its
own field, so a single run can probe many distinct fields.

### `champs2d_gradient` — the reconstructed-gradient family

All five fields read the module-reconstructed element gradient $\nabla b$
(`gradient_b_elements`, Green–Gauss with the boundary-cell B2 correction). The
dataset uses the exactly linear field $b=(x-y,\,x+y)$:

$$\nabla b = \begin{pmatrix} 1 & -1 \\ 1 & 1 \end{pmatrix}, \qquad
\nabla\cdot b = 2, \qquad (\nabla\times b)_z = 2$$

| # | Quantity | Formula | Expected |
|---|---|---|---|
| 1 | magnetic divergence | $\nabla\cdot b=\operatorname{tr}\nabla b$ | $2$ |
| 2 | magnetic curl | $(\nabla\times b)_z$ | $2$ |
| 3 | current density | $J_z=(\nabla\times b)_z/\mu_0$ | $1$ |
| 4 | Lorentz force | $F=J\times b$ | $(-(x+y),\,x-y)$ |
| 5 | magnetic tension | $F_t=\mu_0^{-1}(b\cdot\nabla)b$ | $(-y,\,x)$ |

with $\mu_0=2$ (non-default, so a formula that ignores the `mu0` keyword is
caught). The divergence is probed at an **interior** cell and at a **wall**
cell — the latter exercises the one-sided B2 reconstruction. The choice
$b=(x-y,\,x+y)$ is deliberate: $b=(x,y)$ would give $\nabla\times b=0$ (making
the curl, current and Lorentz checks vacuous) and $b=(-y,x)$ would give
$\nabla\cdot b=0$ (making the divergence check vacuous).

### `champs2d_transport` — the velocity / parameter family

Uniform $b=(3,4)$ and uniform $u=(4,3)$, so $(u\cdot\nabla)b=0$ and
$\nabla^2 b=0$: $b$ is an exact discrete equilibrium and every reference below
is position-independent. Dataset constants $\mu_0=2$, $q/m=2$, $\mu=2$,
$\rho=4$ ($\nu=\mu/\rho=0.5$), $\eta=0.25$:

| # | Quantity | Formula | Expected |
|---|---|---|---|
| 1 | Poynting vector | $S=\mu_0^{-1}(|b|^2u-(u\cdot b)b)$ | $(14,-10.5)$ |
| 2 | crossed helicity | $h_c=u\cdot b$ | $24$ |
| 3 | magnetic Reynolds number | $R_m=|u|L_e/\eta$ | $20\,h$ |
| 4 | magnetic Prandtl number | $P_m=\nu/\eta$ | $2$ |
| 5 | magnetic energy | $e_m=|b|^2/(2\mu_0)$ | $6.25$ |
| 6 | cyclotron pulsation | $\omega_c=(q/m)|b|$ | $10$ |

Every constant is deliberately **non-default**, so a formula with a hard-coded
`1.0` cannot pass. Both vectors have length $5$ exactly and every reference is
an integer or a terminating decimal: a TRUST `.son` probe file stores only 9
significant digits, so a reference such as $2\sqrt{5}$ could not be asserted to
machine precision. $b$ and $u$ are also deliberately neither parallel (the
Poynting vector would vanish) nor orthogonal ($u\cdot b$ would vanish).

In [ ]:
from trustutils import run

run.TRUST_parameters("1.9.8")
run.reset()

## Helpers: read a TRUST probe (`.son`) file

Each dataset runs in its own `build/<dataset>/<scale>/` directory (the merged
form layout), so the readers take the build subdirectory and the case name
explicitly. `probe_pos()` recovers the **effective** probe position from the
`.son` header: the probe is snapped to the nearest cell centre, and on a
refined mesh the nearest centre is not the requested point.

In [ ]:
import numpy as np
import os
from trustutils.files import SonFile

BUILD = run.BUILD_DIRECTORY

def read_son_dir(subdir, case_name, sonde):
    """(time, values) of probe `sonde` of case `case_name` in build/<subdir>/."""
    path = f"{BUILD}/{subdir}/{case_name}_{sonde}.son"
    son = SonFile(path, None)
    entries = son.getEntries()
    assert entries, f"no probe entry found in {path}"
    t, v = son.getValues(entries[0])
    return np.asarray(t, dtype=float), np.asarray(v, dtype=float)

def read_scalar_dir(subdir, case_name, sonde):
    """Final value of a scalar post-treatment field at the probe."""
    t, v = read_son_dir(subdir, case_name, sonde)
    return float(v[-1])

def read_vector_dir(subdir, case_name, sonde):
    """Final value of every component of a vector post-treatment field."""
    path = f"{BUILD}/{subdir}/{case_name}_{sonde}.son"
    son = SonFile(path, None)
    return np.array([float(np.asarray(son.getValues(e)[1], dtype=float)[-1])
                     for e in son.getEntries()])

def probe_pos(path, dim=2):
    """Effective probe position (snapped to a cell centre), from the .son header."""
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            if line.startswith("# Temps"):
                toks = line.replace("=", " = ").split()
                out = [float(toks[toks.index("x") + 2]),
                       float(toks[toks.index("y") + 2])]
                if dim == 3:
                    out.append(float(toks[toks.index("z") + 2]))
                return tuple(out)
    raise RuntimeError(f"no probe position header in {path}")

print("BUILD =", BUILD)

## Run

Two datasets, each with the shared refinement ladder (`coarse` = the baseline
mesh, `standard` = cells ×2, `fine` = cells ×4), each scale in its own
`build/<dataset>/<scale>/` directory. Runs on **2 processors** (`nbProcs=2` +
`case.partition()`); the datasets carry the commented `# BEGIN PARTITION` /
`# BEGIN SCATTER` regions that `trust -partition` toggles.

In [ ]:
from trustutils import run

NB_PROCS = 2

run.reset()
# --- mesh-convergence scan (shared ladder, per-dataset subdirectories) ------
# Each dataset src/<name>.data carries $nx/$ny node-count placeholders; every
# scale runs the SAME file via trustutils substitution
# (run.addCaseFromTemplate -> string.Template) into build/<name>/<scale>/:
#   coarse   = the baseline mesh (11x11)
#   standard = cells x2          (21x21)
#   fine     = cells x4          (41x41)
# Each scale runs in its own build/<name>/<scale>/ directory so that the
# partitioned DOM_000*.Zones of one level never overwrite another's.
LEVELS = (("coarse", 11), ("standard", 21), ("fine", 41))
cases = {}
for base in ("champs2d_gradient", "champs2d_transport"):
    cases[base] = {}
    for lvl, n in LEVELS:
        cases[base][lvl] = run.addCaseFromTemplate(
            f"{base}.data", f"{base}/{lvl}", dic={"nx": n, "ny": n},
            nbProcs=NB_PROCS, excluNR=True)
        cases[base][lvl].partition()
run.printCases()
run.runCases()

## Checks — `champs2d_gradient` (the $\nabla b$ family)

In [ ]:
BASE = "champs2d_gradient"
CN = cases[BASE]["coarse"].dataFileName_
SUB = f"{BASE}/coarse"

div   = read_scalar_dir(SUB, CN, "SONDE_DIV")
divbd = read_scalar_dir(SUB, CN, "SONDE_DIV_BD")
rot   = read_scalar_dir(SUB, CN, "SONDE_ROT")
jz    = read_scalar_dir(SUB, CN, "SONDE_J")
fl    = read_vector_dir(SUB, CN, "SONDE_FL")
ft    = read_vector_dir(SUB, CN, "SONDE_FT")
xy    = probe_pos(f"{BUILD}/{SUB}/{CN}_SONDE_FL.son", dim=2)

exp_fl = np.array([-(xy[0] + xy[1]), xy[0] - xy[1]])
exp_ft = np.array([-xy[1], xy[0]])

print(f"probe                  : {xy}")
print(f"divergence (interior)  : {div:.15g}   expected 2")
print(f"divergence (wall cell) : {divbd:.15g}   expected 2")
print(f"rotationnel            : {rot:.15g}   expected 2")
print(f"courant (J_z)          : {jz:.15g}   expected 1")
print(f"force_lorentz          : {fl}   expected {exp_fl}")
print(f"tension_magnetique     : {ft}   expected {exp_ft}")

assert abs(div   - 2.0) < 1e-12, f"divergence_magnetique wrong: {div}"
assert abs(divbd - 2.0) < 1e-12, f"wall-cell divergence_magnetique wrong: {divbd}"
assert abs(rot   - 2.0) < 1e-12, f"rotationnel_magnetique wrong: {rot}"
assert abs(jz    - 1.0) < 1e-12, f"courant_magnetique wrong: {jz}"
assert np.max(np.abs(fl - exp_fl)) < 1e-12, f"force_lorentz wrong: {fl}"
assert np.max(np.abs(ft - exp_ft)) < 1e-12, f"tension_magnetique wrong: {ft}"
print("OK: the five grad-b fields match their analytic references "
      "(interior and wall cells).")

## Checks — `champs2d_transport` (the velocity / parameter family)

In [ ]:
BASE = "champs2d_transport"
CN = cases[BASE]["coarse"].dataFileName_
SUB = f"{BASE}/coarse"

# dataset parameters (src/champs2d_transport.data)
B   = np.array([3.0, 4.0])           # uniform b
U   = np.array([4.0, 3.0])           # uniform u
MU0, QM, MU, RHO, ETA = 2.0, 2.0, 2.0, 4.0, 0.25
NU  = MU / RHO                       # kinematic viscosity
H   = 1.0 / (11 - 1)                 # baseline (coarse) cell size

S  = read_vector_dir(SUB, CN, "SONDE_POYNTING")
hc = read_scalar_dir(SUB, CN, "SONDE_HC")
rm = read_scalar_dir(SUB, CN, "SONDE_RM")
pm = read_scalar_dir(SUB, CN, "SONDE_PM")
em = read_scalar_dir(SUB, CN, "SONDE_EM")
om = read_scalar_dir(SUB, CN, "SONDE_OMEGA")

exp_S  = (B @ B * U - (U @ B) * B) / MU0
exp_hc = float(U @ B)
exp_rm = float(np.linalg.norm(U) * H / ETA)
exp_pm = NU / ETA
exp_em = float(B @ B) / (2 * MU0)
exp_om = QM * float(np.linalg.norm(B))

print(f"vecteur_poynting              : {S}   expected {exp_S}")
print(f"helicite_croisee              : {hc:.15g}   expected {exp_hc:.15g}")
print(f"nombre_de_reynolds_magnetique : {rm:.15g}   expected {exp_rm:.15g}")
print(f"nombre_de_prandtl_magnetique  : {pm:.15g}   expected {exp_pm:.15g}")
print(f"energie_magnetique            : {em:.15g}   expected {exp_em:.15g}")
print(f"pulsation_cyclotronique       : {om:.15g}   expected {exp_om:.15g}")

assert np.max(np.abs(S - exp_S)) < 1e-12, f"vecteur_poynting wrong: {S}"
assert abs(hc - exp_hc) < 1e-12, f"helicite_croisee wrong: {hc}"
assert abs(rm - exp_rm) / exp_rm < 1e-12, f"nombre_de_reynolds_magnetique wrong: {rm}"
assert abs(pm - exp_pm) < 1e-12, f"nombre_de_prandtl_magnetique wrong: {pm}"
assert abs(em - exp_em) < 1e-12, f"energie_magnetique wrong: {em}"
assert abs(om - exp_om) / exp_om < 1e-12, f"pulsation_cyclotronique wrong: {om}"
print("OK: the six transport fields match their analytic references.")

## Mesh convergence

Both datasets are **discrete identities**: every analytic reference above is
reconstructed exactly from the discrete operators (Green–Gauss is exact for a
linear $b$ at interior cells, and the B2 correction makes it exact at wall
cells; uniform data gives a uniform field), so the deviation must stay at
machine precision at **every** mesh level. No convergence order is reported —
only that the identity does not degrade under refinement.

Note the two level-dependent subtleties the scan deliberately covers:
$L_e = h = 1/(n-1)$ shrinks with the mesh, so the magnetic Reynolds reference
is recomputed per level; and the probe is snapped to the nearest cell centre,
whose coordinates are read back from the `.son` header.

In [ ]:
def err_gradient(sub, cn):
    """Max absolute deviation of the five grad-b fields from their references."""
    xy = probe_pos(f"{BUILD}/{sub}/{cn}_SONDE_FL.son", dim=2)
    errs = [
        abs(read_scalar_dir(sub, cn, "SONDE_DIV")    - 2.0),
        abs(read_scalar_dir(sub, cn, "SONDE_DIV_BD") - 2.0),
        abs(read_scalar_dir(sub, cn, "SONDE_ROT")    - 2.0),
        abs(read_scalar_dir(sub, cn, "SONDE_J")      - 1.0),
        float(np.max(np.abs(read_vector_dir(sub, cn, "SONDE_FL")
                            - np.array([-(xy[0] + xy[1]), xy[0] - xy[1]])))),
        float(np.max(np.abs(read_vector_dir(sub, cn, "SONDE_FT")
                            - np.array([-xy[1], xy[0]])))),
    ]
    return max(errs)

def err_transport(sub, cn, n_nodes):
    """Max absolute deviation of the six transport fields from their references."""
    h = 1.0 / (n_nodes - 1)
    errs = [
        float(np.max(np.abs(read_vector_dir(sub, cn, "SONDE_POYNTING")
                            - (B @ B * U - (U @ B) * B) / MU0))),
        abs(read_scalar_dir(sub, cn, "SONDE_HC") - float(U @ B)),
        abs(read_scalar_dir(sub, cn, "SONDE_RM") - float(np.linalg.norm(U) * h / ETA)),
        abs(read_scalar_dir(sub, cn, "SONDE_PM") - NU / ETA),
        abs(read_scalar_dir(sub, cn, "SONDE_EM") - float(B @ B) / (2 * MU0)),
        abs(read_scalar_dir(sub, cn, "SONDE_OMEGA") - QM * float(np.linalg.norm(B))),
    ]
    return max(errs)

def scan_row(base, lvl, n):
    cn = cases[base][lvl].dataFileName_
    sub = f"{base}/{lvl}"
    if base == "champs2d_gradient":
        return err_gradient(sub, cn)
    return err_transport(sub, cn, n)

print(f"{'dataset':<20} {'level':<9} {'nodes':>6}  {'max |err|':>12}")
for base in ("champs2d_gradient", "champs2d_transport"):
    for lvl, n in LEVELS:
        e = scan_row(base, lvl, n)
        print(f"{base:<20} {lvl:<9} {n:>6}  {e:12.3e}")
        assert e < 1e-12, f"{base} identity broken at {lvl}: {e:.3e}"
print("OK: every analytic identity holds at every mesh level.")

## Computer Performance

In [ ]:
run.tablePerf()